# Fit Signal — walkthrough

How Huckberry's jackets fit, read from the review data the store already collects. Public data only. Run top to bottom; each step is small enough to read and defend.

Pulling raw reviews is in `src/fetch_reviews.py`; here we work from the committed pull.

## 1. Load the flagship reviews

1,500 reviews of the Flint & Tinder waxed trucker, date-sorted so the star-rating mix is representative. ~80% carry a structured **Fit** rating and the reviewer's height, weight, and kept size.

In [ ]:
import json, re, math
from collections import Counter, defaultdict
rows = json.load(open('data/flagship_structured.json'))
print(len(rows), 'reviews'); rows[1]

## 2. Does it run small?

Fit is coded 1 = runs small, 2 = true to size, 3 = runs large (confirmed by cross-checking the code against the words reviewers use).

In [ ]:
fit = [r['f'] for r in rows if r['f'] in (1,2,3)]
c = Counter(fit); n = len(fit)
print(f"n={n}  runs small {100*c[1]/n:.1f}%  true {100*c[2]/n:.1f}%  runs large {100*c[3]/n:.1f}%")
print(f"net runs-small: {100*(c[1]-c[3])/n:+.1f} pp")

Clean 'size up' story. It stops being one story once you split by build.

In [ ]:
for bt in ['Slim','Average','Athletic','Heavy Set']:
    sub=[r['f'] for r in rows if (r['bt'] or '').strip()==bt and r['f'] in (1,2,3)]
    if len(sub)>=20:
        cc=Counter(sub)
        print(f"{bt:10s} runs-small {100*cc[1]/len(sub):4.1f}%  (n={len(sub)})")

Heavy-set reviewers report it small ~3x as often as slim reviewers. A single product-page 'runs small' flag is right for one and wrong for the other.

## 3. Size recommender

Reviewers volunteer height, weight, and the size they kept. Bin by height and weight, and read off the size people like you actually kept.

In [ ]:
def H(h):
    m=re.search(r"(\d)\s*['’′]\s*(\d{1,2})", str(h or ''))
    return int(m.group(1))*12+int(m.group(2)) if m else None
def W(w):
    m=re.search(r'(\d{2,3})', str(w or '')); v=int(m.group(1)) if m else None
    return v if v and 90<=v<=400 else None
SZ={'xs':'XS','s':'S','small':'S','m':'M','medium':'M','l':'L','large':'L','xl':'XL','xxl':'XXL','2xl':'XXL','3xl':'3XL'}
def S(s): return SZ.get(str(s or '').strip().lower().replace(' ',''))
def hb(h): return '≤5ft6' if h<=67 else '5ft7-8' if h<=69 else '5ft9-10' if h<=71 else '5ft11-6ft0' if h<=73 else '6ft1+'
def wb(w): lo=(w//20)*20; return f'{lo}-{lo+19}'
grid=defaultdict(list)
for r in rows:
    h,w,s=H(r['h']),W(r['w']),S(r['sk'])
    if h and w and s: grid[(hb(h),wb(w))].append(s)
cell=grid[('5ft11-6ft0','180-199')]
print('5\'11\"-6\'0\", 180-199 lb  ->', Counter(cell).most_common())

## 4. The AI step, and why to trust it

For brands that don't collect a structured fit rating, fit has to come from the review **text**. That's an LLM read (`src/classify.py`). Before trusting it, score it where truth exists: on the flagship, have the model read only the text, then check it against the structured Fit rating it never saw.

`data/flagship_llm_labels.tsv` holds the recorded labels; `data/classify_source.json` holds the truth.

In [ ]:
src=json.load(open('data/classify_source.json'))['flagship']
truth=[{1:'small',2:'true',3:'large'}[r['f']] for r in src]
lab=[l.split('\t')[1] for l in open('data/flagship_llm_labels.tsv').read().split(chr(10)) if l]
pairs=list(zip(lab,truth)); sig=[(p,t) for p,t in pairs if p!='none']
print(f"no fit cue in text: {sum(p=='none' for p in lab)/len(lab):.0%}")
print(f"accuracy where text mentions fit: {sum(p==t for p,t in sig)/len(sig):.1%} (n={len(sig)})")

Half of reviews never mention fit even though the shopper recorded it in the field. Where fit is mentioned, the model is reliable, and its brand-level read matches the structured truth (83% vs 85% net-small on the flagship). That match is what licenses extending it to Relwen and Barbour.

## 5. Cross-brand

`src/pipeline.py` applies the same text read to all three brands. Share of fit-mentioning reviews that say 'runs small':

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'src/pipeline.py'])
r=json.load(open('data/results_final.json'))
for b,v in r['crossbrand'].items():
    print(f"{b:16s} {v['pct_small_of_directional']}%  (n={v['directional']}, CI {v['ci_small']})")

All three lean small, but differently: Flint & Tinder is near one-directional, Barbour is contested, Relwen sits between on a thin sample. Read them as direction, not decimals.

The real outcome is returns, which live in Huckberry's systems. Pointed at that, this becomes a dollar figure instead of an inference.